# Monitor - Log Pattern Analysis Agent

In this exercise, we build an agent that reads a folder of raw service logs, spots patterns and anomalies, and proposes Prometheus / Alertmanager monitoring rules to catch those situations going forward.

The notebook seeds a tiny demo log set so it is runnable out-of-the-box; point `logs_folder` at your own logs to try it on real data.

## Environment Variables & Imports

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GOOGLE_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTRHOPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>
```


In [ ]:
import initialize_notebook # noqa

In [ ]:
import json
import os
import pathlib

import jinja2

from hslu.dlm03.common import backend as backend_lib
from hslu.dlm03.common import chat as chat_lib
from hslu.dlm03.common import presets
from hslu.dlm03.common.displays import ipython_display
from hslu.dlm03.common import tools as tools_lib
from hslu.dlm03.util import file as file_lib

## Parameters

In [ ]:
backend_name = "Gemma 4 26B"
backend_config = presets.CONFIGS_BY_NAME[backend_name]
backend = backend_lib.LLM.from_config(backend_config)

In [ ]:
# Folder containing service logs to analyze (one file per service, plain text).
logs_folder = pathlib.Path("/tmp/dlm03-logs/")
# Folder where the agent can write proposed monitoring rules (Prometheus / Alertmanager YAML).
rules_folder = pathlib.Path("/tmp/dlm03-monitoring-rules/")
rules_folder.mkdir(parents=True, exist_ok=True)

# Seed a small demo log set so the exercise is runnable out-of-the-box.
def _seed_demo_logs() -> None:
    logs_folder.mkdir(parents=True, exist_ok=True)
    if any(logs_folder.iterdir()):
        return
    (logs_folder / "checkout.log").write_text(
        """2026-04-23T08:01:12Z INFO  checkout order=ord_001 status=paid latency=230ms
2026-04-23T08:01:14Z INFO  checkout order=ord_002 status=paid latency=210ms
2026-04-23T08:02:51Z WARN  checkout payment_gateway timeout after 3000ms (retry 1/3)
2026-04-23T08:02:55Z WARN  checkout payment_gateway timeout after 3000ms (retry 2/3)
2026-04-23T08:02:58Z ERROR checkout payment_gateway timeout after 3000ms (retry 3/3) - giving up
2026-04-23T08:03:02Z ERROR checkout order=ord_003 status=failed reason=gateway_timeout
2026-04-23T08:03:14Z ERROR checkout order=ord_004 status=failed reason=gateway_timeout
2026-04-23T08:03:30Z ERROR checkout order=ord_005 status=failed reason=gateway_timeout
2026-04-23T08:10:10Z INFO  checkout order=ord_006 status=paid latency=245ms
"""
    )
    (logs_folder / "auth.log").write_text(
        """2026-04-23T08:00:01Z INFO  auth login user=u_001 result=ok latency=80ms
2026-04-23T08:00:05Z INFO  auth login user=u_002 result=ok latency=72ms
2026-04-23T08:05:10Z WARN  auth token_cache miss ratio=0.43
2026-04-23T08:05:30Z WARN  auth token_cache miss ratio=0.51
2026-04-23T08:06:00Z WARN  auth token_cache miss ratio=0.62
"""
    )

_seed_demo_logs()

# MCP Server

The server exposes read-only log inspection tools (list, read, grep, count by level) and a `write_monitoring_rule` tool to persist the agent's proposals.

In [ ]:
import re as _re
import subprocess

from mcp.server import FastMCP

SERVER = FastMCP()

LOGS_FOLDER = logs_folder.resolve()
RULES_FOLDER = rules_folder.resolve()


@SERVER.tool()
def list_log_files() -> list[str]:
    """List the log files available for analysis."""
    return [p.name for p in LOGS_FOLDER.glob("*") if p.is_file()]


@SERVER.tool()
def read_log_file(filename: str, max_lines: int = 500) -> str:
    """Read up to `max_lines` lines of a log file.

    For large logs, prefer `grep_log_file` to narrow down.
    """
    path = LOGS_FOLDER / filename
    if not path.exists():
        return "File not found."
    lines = path.read_text().splitlines()
    if len(lines) > max_lines:
        return "\n".join(lines[-max_lines:]) + f"\n\n... (showing last {max_lines} of {len(lines)} lines)"
    return "\n".join(lines)


@SERVER.tool()
def grep_log_file(filename: str, pattern: str, context: int = 2) -> str:
    """Grep a log file for the given regex pattern, returning matches with context lines."""
    path = LOGS_FOLDER / filename
    if not path.exists():
        return "File not found."
    try:
        regex = _re.compile(pattern)
    except _re.error as e:
        return f"Invalid regex: {e}"
    lines = path.read_text().splitlines()
    out: list[str] = []
    for i, line in enumerate(lines):
        if regex.search(line):
            lo = max(0, i - context)
            hi = min(len(lines), i + context + 1)
            out.append("\n".join(lines[lo:hi]))
            out.append("--")
    if not out:
        return "No matches."
    return "\n".join(out)[-5000:]


@SERVER.tool()
def count_log_levels(filename: str) -> dict:
    """Count occurrences of common log levels in a file (INFO/WARN/ERROR/DEBUG)."""
    path = LOGS_FOLDER / filename
    if not path.exists():
        return {"error": "File not found."}
    text = path.read_text()
    return {
        level: len(_re.findall(rf"\b{level}\b", text))
        for level in ("DEBUG", "INFO", "WARN", "ERROR")
    }


@SERVER.tool()
def write_monitoring_rule(filename: str, content: str) -> str:
    """Write a proposed Prometheus / Alertmanager rule file (YAML or PromQL) to disk."""
    if "/" in filename or ".." in filename:
        return "filename must be a simple name inside the rules folder."
    path = RULES_FOLDER / filename
    path.write_text(content)
    return f"Wrote {path}"

In [ ]:
import threading

import uvicorn

PORT = 5000
HOST = "localhost"

RUN_ARGS = {
    "app": SERVER.streamable_http_app,
    "port": PORT,
    "host": HOST,
    "log_level": "warning",
}

MCP_THREAD = threading.Thread(target=uvicorn.run, kwargs=RUN_ARGS, daemon=True)
MCP_THREAD.start()

MCP_URL = f"http://{HOST}:{PORT}/mcp"

# Agentic Harness

In [ ]:
import re
def remove_thoughts(text: str) -> str:
    # Remove well-formed <thought>...</thought> blocks (multiline safe)
    cleaned = re.sub(r"<thought>.*?</thought>", "", text, flags=re.DOTALL | re.IGNORECASE)

    # Optionally handle unclosed <thought> tags (strip until end)
    cleaned = re.sub(r"<thought>.*$", "", cleaned, flags=re.DOTALL | re.IGNORECASE)

    return cleaned.strip()

In [ ]:
system_instructions_template_string = \
"""You are an expert Site Reliability Engineer specialized in observability.

Given a folder of raw service logs, your goal is to:
1. Survey the logs (`list_log_files`, `count_log_levels`, `read_log_file`, `grep_log_file`)
   and build a picture of what services are doing.
2. Identify noteworthy patterns and anomalies - repeated errors, error bursts, latency
   creep, cache-miss ratios, retry storms, quiet services that used to be loud, etc.
3. For each meaningful pattern, propose a **Prometheus/Alertmanager** rule that would catch
   this situation going forward. Every rule should include:
   - a sensible `alert` name,
   - a PromQL `expr` (you may assume typical metric names like
     `http_requests_total`, `request_duration_seconds_bucket`, `cache_miss_total`, ...),
   - a `for:` window that avoids flapping,
   - `labels` (at minimum `severity`) and `annotations` (`summary`, `description`, optional
     `runbook_url`).
4. Write each proposal as a separate YAML file via `write_monitoring_rule` (one
   `alert`/file, or a grouped `groups:` file - your call, but keep it valid Prometheus YAML).
5. End with a short human-readable summary listing the patterns you spotted and the rules
   you wrote.
"""
system_instructions_template = jinja2.Template(system_instructions_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message_template_string = \
"""Please analyze the logs under {{ logs_folder }} and propose Prometheus monitoring rules
that would catch the problems you find. Write each rule file into {{ rules_folder }}."""
user_message_template = jinja2.Template(user_message_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
system_instructions = system_instructions_template.render()
user_message = user_message_template.render(
    logs_folder=logs_folder, rules_folder=rules_folder,
)

chat_display = ipython_display.IPythonChatDisplay()
chat_display.show()
chat = chat_lib.Chat(
    messages=[
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_message},
    ],
    observers=[chat_display],
)
async with tools_lib.mcp_session(MCP_URL) as session:
    mcp_tools = await session.list_tools()
    tools = [tools_lib.tool_from_mcp(tool) for tool in mcp_tools.tools]
    done = False
    while not done:
        response = backend.generate(chat, tools=tools)
        done = True
        message = response.choices[0].message
        if message.content is not None:
            message.content = remove_thoughts(message.content)
        chat.append(message)
        for tool_call in message.tool_calls or ():
            done = False
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            tool_call_result = await session.call_tool(tool_name, arguments)
            for content in tool_call_result.content:
                tool_call_result_content = tools_lib.tool_call_result_from_mcp(
                    tool_call.id,
                    content,
                )
                chat.append(tool_call_result_content)

## Generated rules

In [ ]:
for rule_file in sorted(rules_folder.glob("*")):
    print(f"=== {rule_file.name} ===")
    print(rule_file.read_text())
    print()